### Pinecone

In [9]:
from langchain_teddynote.korean import stopwords
from langchain_teddynote import logging
from dotenv import load_dotenv

logging.langsmith("test0914")
load_dotenv()

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


True

In [10]:
stopword = stopwords()
stopword[:20]

['아',
 '휴',
 '아이구',
 '아이쿠',
 '아이고',
 '어',
 '나',
 '우리',
 '저희',
 '따라',
 '의해',
 '을',
 '를',
 '에',
 '의',
 '가',
 '으로',
 '로',
 '에게',
 '뿐이다']

In [1]:
!uv pip install pymupdf

Resolved 1 package in 226ms
 Downloaded pymupdf
Prepared 1 package in 4.15s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 156ms
 + pymupdf==1.28.2


In [11]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
import glob

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
split_docs = []


files = sorted(glob.glob("*.pdf"))

for file in files:
    loader = PyMuPDFLoader(file)
    split_docs.extend(loader.load_and_split(text_splitter))

len(split_docs)

119

In [12]:
split_docs[0].metadata

{'producer': 'Hancom PDF 1.3.0.542',
 'creator': 'Hwp 2018 10.0.0.13462',
 'creationdate': '2023-12-08T13:28:38+09:00',
 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'total_pages': 23,
 'format': 'PDF 1.4',
 'title': '',
 'author': 'dj',
 'subject': '',
 'keywords': '',
 'moddate': '2023-12-08T13:28:38+09:00',
 'trapped': '',
 'modDate': "D:20231208132838+09'00'",
 'creationDate': "D:20231208132838+09'00'",
 'page': 0}

In [13]:
from langchain_teddynote.community.pinecone import preprocess_documents

contents, metadatas = preprocess_documents(
    split_docs= split_docs,
    metadata_keys=["source", "page", "author"],
    min_length=5,
    use_basename= True
)

  0%|          | 0/119 [00:00<?, ?it/s]

In [14]:
contents[:5]

['2023년 12월호',
 '2023년 12월호\nⅠ. 인공지능 산업 동향 브리프\n 1. 정책/법제 \n   ▹ 미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표  ························· 1\n   ▹ G7, 히로시마 AI 프로세스를 통해 AI 기업 대상 국제 행동강령에 합의··························· 2\n   ▹ 영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언··························· 3',
 '▹ 미국 법원, 예술가들이 생성 AI 기업에 제기한 저작권 소송 기각····································· 4\n   ▹ 미국 연방거래위원회, 저작권청에 소비자 보호와 경쟁 측면의 AI 의견서 제출················· 5\n   ▹ EU AI 법 3자 협상, 기반모델 규제 관련 견해차로 난항··················································· 6\n \n 2. 기업/산업',
 '2. 기업/산업 \n   ▹ 미국 프런티어 모델 포럼, 1,000만 달러 규모의 AI 안전 기금 조성································ 7\n   ▹ 코히어, 데이터 투명성 확보를 위한 데이터 출처 탐색기 공개  ······································· 8\n   ▹ 알리바바 클라우드, 최신 LLM ‘통이치엔원 2.0’ 공개 ······················································ 9',
 '▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10\n   ▹ 구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 ···········································

In [15]:
metadatas["source"][:5]

['SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf']

### 새로운 벡터 스토어 인덱스 생성하기

In [16]:
import os
from langchain_teddynote.community.pinecone import create_index

pc_index = create_index(
    api_key=os.environ["PINECONE_API_KEY"],
    index_name="teddynote-db-index",
    dimension=4096,
    metric="dotproduct"
)

[create_index]
DescribeIndexStatsResponse(dimension=4096, total_vector_count=0, metric='dotproduct', namespaces=0)


### 희소 인코더 생성하기
- 희소 인코더는 텍스트 데이터를 희소 벡터로 변환
- 희소 벡터란 대부분의 값이 0이고, 소수의 값만 중요한 의미를 가지는 고차원 벡터

In [17]:
from langchain_teddynote.community.pinecone import (
    create_sparse_encoder, fit_sparse_encoder
)

sparse_encoder = create_sparse_encoder(stopwords(), mode= "kiwi")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [18]:
saved_path = fit_sparse_encoder(
    sparse_encoder= sparse_encoder,
    contents= contents,
    save_path="./sparse_encoder.pkl"
)

  0%|          | 0/119 [00:00<?, ?it/s]

[fit_sparse_encoder]
Saved Sparse Encoder to: ./sparse_encoder.pkl


In [19]:
# 저장된 .pkl 파일은 리트리버에 활용
# 나중에 학습하고 저장한 희소 인코더를 다시 불러올 때는 아래 코드

from langchain_teddynote.community.pinecone import load_sparse_encoder

sparse_encoder = load_sparse_encoder("./sparse_encoder.pkl")

[load_sparse_encoder]
Loaded Sparse Encoder from: ./sparse_encoder.pkl


### Pinecone 데이터베이스 인덱스에 문서 추가하기

In [29]:
from langchain_openai import OpenAIEmbeddings
from langchain_upstage import UpstageEmbeddings

openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
upstage_embeddings = UpstageEmbeddings(model="solar-embedding-1-large-passage")

In [30]:
%%time

from langchain_teddynote.community.pinecone import upsert_documents
from langchain_upstage import UpstageEmbeddings

upsert_documents(
    index= pc_index,
    namespace= "teddynote-namespace-01",
    contents= contents,
    metadatas= metadatas,
    sparse_encoder= sparse_encoder,
    embedder = upstage_embeddings,
    batch_size= 32
)


  0%|          | 0/4 [00:00<?, ?it/s]

[upsert_documents]
DescribeIndexStatsResponse(dimension=4096, total_vector_count=119, metric='dotproduct', namespaces=1)
CPU times: total: 1.34 s
Wall time: 11 s


In [ ]:
%%time

from langchain_teddynote.community.pinecone import upsert_documents_parallel

upsert_documents_parallel(
    index= pc_index,
    namespace= "teddynote-namespace-02",
    contents= contents,
    metadatas= metadatas,
    sparse_encoder= sparse_encoder,
    embedder= upstage_embeddings,
    batch_size= 64,
    max_workers= 30
)

문서 Upsert 중:   0%|          | 0/4 [00:00<?, ?it/s]

Upsert 중 오류 발생: GrpcIndex.upsert() got an unexpected keyword argument 'async_req'
Upsert 중 오류 발생: GrpcIndex.upsert() got an unexpected keyword argument 'async_req'
Upsert 중 오류 발생: GrpcIndex.upsert() got an unexpected keyword argument 'async_req'
Upsert 중 오류 발생: GrpcIndex.upsert() got an unexpected keyword argument 'async_req'
총 0개의 Vector 가 Upsert 되었습니다.
DescribeIndexStatsResponse(dimension=4096, total_vector_count=119, metric='dotproduct', namespaces=1)
CPU times: total: 1.17 s
Wall time: 3.9 s


### 인덱스 조회 및 삭제하기

In [23]:
pc_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=4096, total_vector_count=119, metric='dotproduct', namespaces=1)

In [24]:
# 특정 네임스페이스에 저장된 데이터를 삭제가능
from langchain_teddynote.community.pinecone import delete_namespace

delete_namespace(
    pinecone_index= pc_index,
    namespace= "teddynote-namespace-01"
)

네임스페이스 'teddynote-namespace-01'의 모든 데이터가 삭제되었습니다.


In [25]:
pc_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=4096, total_vector_count=0, metric='dotproduct', namespaces=0)

### 리트리버 생성하기

In [35]:
import os

from langchain_teddynote.korean import stopwords
from langchain_teddynote.community.pinecone import init_pinecone_index
from langchain_upstage import UpstageEmbeddings

pinecone_params = init_pinecone_index(
    index_name= "teddynote-db-index",
    namespace="teddynote-namespace-01",
    api_key= os.environ["PINECONE_API_KEY"],
    sparse_encoder_path= "./sparse_encoder.pkl",
    stopwords= stopwords(),
    tokenizer= "kiwi",
    embeddings= UpstageEmbeddings(model="solar-embedding-1-large-query"),
    top_k=5,
    alpha= 0.5
)

[init_pinecone_index]
DescribeIndexStatsResponse(dimension=4096, total_vector_count=119, metric='dotproduct', namespaces=1)


In [36]:
from langchain_teddynote.community.pinecone import PineconeKiwiHybridRetriever

pinecone_retriever = PineconeKiwiHybridRetriever(**pinecone_params)

In [37]:
search_results = pinecone_retriever.invoke("gpt-4o 미니 출시 관련 정보에 대해서 알려줘")

for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n=================\n")

생성에서 가장 우수한 성능을 발휘
KEY Contents
£ 주요 LLM 중 GPT-4가 가장 환각 현상 적고 GPT-3.5 터보도 비슷한 성능 기록
n 머신러닝 데이터 관리 기업 갈릴레오(Galileo)가 2023년 11월 15일 주요 LLM의 환각 현상을 평가한 
‘LLM 환각 지수(LLM Hallucination Index)’를 발표
∙생성 AI의 환각 현상은 AI 시스템이 잘못된 정보를 생성하거나, 현실과 다른 부정확한 결과를 내놓는
{'author': 'dj', 'context': '생성에서 가장 우수한 성능을 발휘\nKEY Contents\n£ 주요 LLM 중 GPT-4가 가장 환각 현상 적고 GPT-3.5 터보도 비슷한 성능 기록\nn 머신러닝 데이터 관리 기업 갈릴레오(Galileo)가 2023년 11월 15일 주요 LLM의 환각 현상을 평가한 \n‘LLM 환각 지수(LLM Hallucination Index)’를 발표\n∙생성 AI의 환각 현상은 AI 시스템이 잘못된 정보를 생성하거나, 현실과 다른 부정확한 결과를 내놓는', 'page': 19.0, 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}


▹ 구글 딥마인드, 범용 AI 모델의 기능과 동작에 대한 분류 체계 발표······························ 16
   ▹ 갈릴레오의 LLM 환각 지수 평가에서 GPT-4가 가장 우수 ··········································· 17
   
 4. 인력/교육     
   ▹ 영국 옥스퍼드 인터넷 연구소, AI 기술자의 임금이 평균 21% 높아······························· 18
   
   
 
Ⅱ. 주요 행사
{'author': 'dj', 'context': '▹ 구글 딥마인드, 범용 AI 모델의 기능과 동작에 대한 분류 체계 발표······························ 16\n   ▹ 갈릴레오의 LLM 환각 지수 평가에서 G

In [38]:
search_results = pinecone_retriever.invoke(
    "gpt-4o 미니 출시 관련 정보에 대해서 알려줘", search_kwargs={"k": 1}
)

for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n===================\n")

생성에서 가장 우수한 성능을 발휘
KEY Contents
£ 주요 LLM 중 GPT-4가 가장 환각 현상 적고 GPT-3.5 터보도 비슷한 성능 기록
n 머신러닝 데이터 관리 기업 갈릴레오(Galileo)가 2023년 11월 15일 주요 LLM의 환각 현상을 평가한 
‘LLM 환각 지수(LLM Hallucination Index)’를 발표
∙생성 AI의 환각 현상은 AI 시스템이 잘못된 정보를 생성하거나, 현실과 다른 부정확한 결과를 내놓는
{'author': 'dj', 'context': '생성에서 가장 우수한 성능을 발휘\nKEY Contents\n£ 주요 LLM 중 GPT-4가 가장 환각 현상 적고 GPT-3.5 터보도 비슷한 성능 기록\nn 머신러닝 데이터 관리 기업 갈릴레오(Galileo)가 2023년 11월 15일 주요 LLM의 환각 현상을 평가한 \n‘LLM 환각 지수(LLM Hallucination Index)’를 발표\n∙생성 AI의 환각 현상은 AI 시스템이 잘못된 정보를 생성하거나, 현실과 다른 부정확한 결과를 내놓는', 'page': 19.0, 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}




In [39]:
search_results = pinecone_retriever.invoke(
    "앤스로픽", search_kwargs= {"alpha": 1, "k": 1}
)

for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n===================\n")

£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공
n 구글이 2023년 10월 27일 앤스로픽에 최대 20억 달러를 투자하기로 합의했으며, 이 중 5억 
달러를 우선 투자하고 향후 15억 달러를 추가로 투자할 방침
∙구글은 2023년 2월 앤스로픽에 이미 5억 5,000만 달러를 투자한 바 있으며, 아마존도 지난 9월 
앤스로픽에 최대 40억 달러의 투자 계획을 공개
∙한편, 2023년 11월 8일 블룸버그 보도에 따르면 앤스로픽은 구글의 클라우드 서비스 사용을 위해
{'author': 'dj', 'context': '£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공\nn 구글이 2023년 10월 27일 앤스로픽에 최대 20억 달러를 투자하기로 합의했으며, 이 중 5억 \n달러를 우선 투자하고 향후 15억 달러를 추가로 투자할 방침\n∙구글은 2023년 2월 앤스로픽에 이미 5억 5,000만 달러를 투자한 바 있으며, 아마존도 지난 9월 \n앤스로픽에 최대 40억 달러의 투자 계획을 공개\n∙한편, 2023년 11월 8일 블룸버그 보도에 따르면 앤스로픽은 구글의 클라우드 서비스 사용을 위해', 'page': 13.0, 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}




In [40]:
search_results = pinecone_retriever.invoke(
    "앤스로픽", search_kwargs= {"alpha": 0, "k": 1}
)

for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n===================\n")

1. 정책/법제  
2. 기업/산업 
3. 기술/연구 
 4. 인력/교육
구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 
n 구글이 앤스로픽에 최대 20억 달러 투자에 합의하고 5억 달러를 우선 투자했으며, 앤스로픽은 
구글과 클라우드 서비스 사용 계약도 체결
n 3대 클라우드 사업자인 구글, 마이크로소프트, 아마존은 차세대 AI 모델의 대표 기업인 
앤스로픽 및 오픈AI와 협력을 확대하는 추세
KEY Contents
£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공
{'author': 'dj', 'context': '1. 정책/법제  \n2. 기업/산업 \n3. 기술/연구 \n 4. 인력/교육\n구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 \nn 구글이 앤스로픽에 최대 20억 달러 투자에 합의하고 5억 달러를 우선 투자했으며, 앤스로픽은 \n구글과 클라우드 서비스 사용 계약도 체결\nn 3대 클라우드 사업자인 구글, 마이크로소프트, 아마존은 차세대 AI 모델의 대표 기업인 \n앤스로픽 및 오픈AI와 협력을 확대하는 추세\nKEY Contents\n£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공', 'page': 13.0, 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}




In [41]:
search_results = pinecone_retriever.invoke(
    "앤스로픽의 claude 출시 관련 내용을 알려줘",
    search_kwargs= {"filter": {"page":{"$lt": 5}}, "k": 2}
)

for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n===================\n")

▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10
   ▹ 구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 ················································ 11
   ▹ IDC, 2027년 AI 소프트웨어 매출 2,500억 달러 돌파 전망··········································· 12
{'author': 'dj', 'context': '▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10\n   ▹ 구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 ················································ 11\n   ▹ IDC, 2027년 AI 소프트웨어 매출 2,500억 달러 돌파 전망··········································· 12', 'page': 1.0, 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}


이해관계자 협의를 통해 필요에 따라 개정할 예정
∙첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 평가 및 완화하는 조치를 채택하고, 
첨단 AI 시스템의 출시와 배포 이후 취약점과 오용 사고, 오용 유형을 파악해 완화
∙첨단 AI 시스템의 성능과 한계를 공개하고 적절하거나 부적절한 사용영역을 알리는 방법으로 투명성을 
보장하고 책임성을 강화
∙산업계, 정부, 시민사회, 학계를 포함해 첨단 AI 시스템을 개발하는 조직 간 정보공유와 사고 발생 시
{'author': 'dj', 'context': '이해관계자 협의를 통해 필요에 따라 개정할 예정\n∙첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 

In [42]:
search_results = pinecone_retriever.invoke(
    "앤스로픽의 claude 3.5 출시 관련 내용을 알려줘",
    search_kwargs={
        "filter" : {"source" : {"$eq": "SPRi AI Brief_8월호_산업동향.pdf"}}, "k" : 3
    }
)

for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n===================\n")